# 23.07 - NLP support: text basics

**Notebook type:** Practice notebook with theory, exercises, TODO cells, smoke checks, and test cases.

**Daily output:** Text I/O mini notebook.

Build a small, reliable text pipeline that preserves Vietnamese characters, normalizes noisy spacing, tokenizes at several levels, and reads a UTF-8 CSV without silent corruption.

## Core Ideas

- **Unicode text and UTF-8 are different layers.** Unicode defines characters; UTF-8 defines how those characters are stored as bytes. Always state the encoding when reading or writing files.
- **Preserve information by default.** Vietnamese diacritics distinguish words, so a support pipeline should not remove them merely to simplify matching.
- **Normalize only what the task needs.** Here we remove a possible byte-order mark, use `casefold()`, collapse whitespace, and standardize spaces around common punctuation.
- **Regex should be Unicode-aware.** Python's default string regex behavior recognizes Unicode letters. A pattern based on `[^\W_]` keeps letters and digits while excluding punctuation and underscores.
- **Tokenization has levels.** Character, word, and sentence tokens answer different questions. Inspect each representation before choosing one for a model.
- **Validate the schema after decoding.** A file can decode successfully and still contain missing or renamed columns.

## Setup and Prepared UTF-8 Data

The setup cell creates a tiny Vietnamese CSV for repeatable practice. Data preparation is provided; it is not a learner TODO.

**Return structure — `prepare_day23_csv`:** Returns one `str`, the relative path `_day23_text_data/vietnamese_samples.csv`. The function creates the parent directory when needed and writes a UTF-8 CSV with exactly three columns: `id` (integer identifier), `raw_text` (Unicode text), and `category` (string label). The file-writing side effect communicates success together with the returned path.

In [ ]:
import os
import re
import pandas as pd

DATA_DIR = "_day23_text_data"
CSV_PATH = os.path.join(DATA_DIR, "vietnamese_samples.csv")


def prepare_day23_csv():
    os.makedirs(DATA_DIR, exist_ok=True)
    records = pd.DataFrame(
        {
            "id": [1, 2, 3],
            "raw_text": [
                "  Xin CHÀO,\tViệt Nam!  ",
                "Dữ liệu UTF-8\ncần được đọc đúng.",
                "Máy học — thị giác và ngôn ngữ.",
            ],
            "category": ["greeting", "data", "ai"],
        }
    )
    records.to_csv(CSV_PATH, index=False, encoding="utf-8")
    return CSV_PATH


prepared_csv_path = prepare_day23_csv()
print("Prepared UTF-8 CSV:", prepared_csv_path)

## Exercise 23-A: Normalize text without losing Vietnamese characters

Implement `normalize_text(text)`. Validate that the input is a string, remove a leading Unicode byte-order-mark character if present, apply `casefold()`, collapse all whitespace to one space, remove spaces before `, . ; : ! ?`, and ensure one space after those punctuation marks when more text follows. Preserve Vietnamese diacritics.

**Return structure — `normalize_text`:** Returns one Python `str`. It is stripped, case-folded Unicode text with internal whitespace collapsed and common punctuation spacing normalized. The function raises `TypeError` for non-string input; it never returns `None`.

In [ ]:
# TODO 23-A
def normalize_text(text):
    # TODO: validate, remove a possible BOM, case-fold, and normalize spacing.
    raise NotImplementedError("Implement normalize_text")


# Smoke check: run this after implementing the function above.
normalized_smoke = normalize_text("  Xin CHÀO,\tViệt Nam!  ")
print("Normalized:", normalized_smoke)

## Exercise 23-B: Tokenize at character, word, and sentence levels

Implement `tokenize_text_levels(text)`. Normalize first, then create character tokens, Unicode-aware word tokens, and simple sentence tokens split after `.`, `!`, or `?`. The word pattern should keep Vietnamese letters and may keep an internal apostrophe or hyphen.

**Return structure — `tokenize_text_levels`:** Returns a `dict` with exactly four keys. `normalized` is one `str`; `characters` is a `list[str]` of length equal to `len(normalized)` with one Unicode character per item; `words` is a `list[str]` containing zero or more word tokens; and `sentences` is a `list[str]` containing zero or more non-empty sentence strings in source order.

In [ ]:
# TODO 23-B
def tokenize_text_levels(text):
    # TODO: call normalize_text and build all four dictionary fields.
    raise NotImplementedError("Implement tokenize_text_levels")


# Smoke check: run this after implementing the function above.
tokens_smoke = tokenize_text_levels("Xin chào Việt Nam. Tôi học AI!")
print("Word tokens:", tokens_smoke["words"])
print("Sentence tokens:", tokens_smoke["sentences"])

## Exercise 23-C: Read and validate a UTF-8 text CSV

Implement `read_utf8_text_csv(csv_path)`. Check that the file exists, read it with `pandas.read_csv(..., encoding="utf-8")`, require the columns `id`, `raw_text`, and `category`, and return those columns in that order. Raise a clear error when the path or schema is wrong.

**Return structure — `read_utf8_text_csv`:** Returns a `pandas.DataFrame` of shape `[N, 3]`, where `N >= 0`. Columns are ordered as `id`, `raw_text`, `category`; each row represents one text record. `id` contains integer-like identifiers, while `raw_text` and `category` contain Python strings. The function raises `FileNotFoundError` for a missing path and `ValueError` listing missing required columns for an invalid schema.

In [ ]:
# TODO 23-C
def read_utf8_text_csv(csv_path):
    # TODO: validate the path, decode explicitly as UTF-8, and validate columns.
    raise NotImplementedError("Implement read_utf8_text_csv")


# Smoke check: run this after implementing the function above.
loaded_smoke = read_utf8_text_csv(prepared_csv_path)
print(loaded_smoke[["id", "raw_text"]].head())

## Exercise 23-D: Build a compact text I/O report

Implement `build_text_io_report(frame)`. Copy the input, normalize every `raw_text`, count word tokens, and count normalized characters. Do not mutate the caller's DataFrame.

**Return structure — `build_text_io_report`:** Returns a new `pandas.DataFrame` of shape `[N, 6]`. It preserves `id`, `raw_text`, and `category`, then adds `normalized_text` (`str`), `word_count` (non-negative integer), and `character_count` (non-negative integer). Row order is unchanged, and the input DataFrame is not modified.

In [ ]:
# TODO 23-D
def build_text_io_report(frame):
    # TODO: copy the frame and add normalized text plus word/character counts.
    raise NotImplementedError("Implement build_text_io_report")


# Smoke check: run this after implementing the function above.
report_smoke = build_text_io_report(loaded_smoke)
print(report_smoke[["id", "normalized_text", "word_count"]])

## Test Cases

Run this cell after completing the TODO cells. A correct implementation prints `Day 23 tests passed`.

**Return structure — `run_day23_tests`:** Returns `None`. Success is communicated by completing all assertions and printing exactly `Day 23 tests passed`; a failed requirement raises `AssertionError`.

In [ ]:
def run_day23_tests():
    assert os.path.exists(prepared_csv_path), "Prepared CSV was not created"

    normalized = normalize_text("\ufeff  Xin CHÀO ,Việt Nam!  ")
    assert normalized == "xin chào, việt nam!"
    assert "à" in normalized and "ệ" in normalized, "Vietnamese diacritics were lost"

    tokenized = tokenize_text_levels("Xin chào Việt Nam. Tôi học AI!")
    assert set(tokenized) == {"normalized", "characters", "words", "sentences"}
    assert tokenized["words"] == ["xin", "chào", "việt", "nam", "tôi", "học", "ai"]
    assert len(tokenized["characters"]) == len(tokenized["normalized"])
    assert tokenized["sentences"] == ["xin chào việt nam.", "tôi học ai!"]

    loaded = read_utf8_text_csv(prepared_csv_path)
    assert loaded.shape == (3, 3)
    assert list(loaded.columns) == ["id", "raw_text", "category"]
    assert "Việt Nam" in loaded.loc[0, "raw_text"]

    original_columns = list(loaded.columns)
    report = build_text_io_report(loaded)
    assert list(loaded.columns) == original_columns, "Input DataFrame was mutated"
    assert report.shape == (3, 6)
    assert list(report.columns) == [
        "id", "raw_text", "category", "normalized_text", "word_count", "character_count"
    ]
    assert report["word_count"].tolist() == [4, 7, 7]
    assert (report["character_count"] > 0).all()
    print("Day 23 tests passed")


run_day23_tests()

## Day 23 Checklist

- [ ] I can explain the difference between Unicode characters and UTF-8 bytes.
- [ ] I preserve Vietnamese diacritics during normalization.
- [ ] I can use Unicode-aware regex for basic word and sentence tokenization.
- [ ] I explicitly specify UTF-8 when reading and writing CSV files.
- [ ] I validate required columns after loading text data.
- [ ] All smoke checks and `run_day23_tests()` pass.